# L5c Example: Apples, Oranges, and Linear Allocation

Suppose we have a fixed budget to spend on apples and oranges. Each fruit has a price and a utility per unit, where utility measures the satisfaction we gain from consuming it. How should we divide our budget to obtain the greatest total utility? We will use a linear model, which assumes that each additional unit of a fruit contributes the same amount of utility.

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
>
> * **Formulate a resource-allocation problem:** Define the fruit quantities as decision variables, construct a linear utility objective, and express the budget and nonnegativity constraints.
> * **Predict optimal allocations:** Compare utility per dollar to determine when we should purchase only apples, only oranges, or any mixture that uses the full budget.
> * **Verify and interpret a solution:** Check expenditure and attained utility, and distinguish a unique optimal allocation from multiple allocations with the same optimal utility.

In this example, we hold prices and the budget fixed and solve three cases with different utility coefficients. We first formulate the model and predict each allocation, then compare the solver results, and finally verify the solutions and explain the case with multiple optima.


___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file defines local paths, activates the course environment, and loads the required packages and course library.

Let's set up our code environment:


In [1]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"))


See the [Julia documentation](https://docs.julialang.org/en/v1/) and the [course library documentation](../../../docs/src/index.md) for the functions and types used here.


___

## Task 1: Formulate the allocation model and predict its solution

In this task, we formulate the fruit-allocation problem and predict each optimal purchase so we can check the solver results.

Let $x_A$ and $x_O$ denote the quantities of apples and oranges purchased; fractional amounts are allowed. The remaining model quantities are:

* $u_A,u_O>0$: constant utility gained per unit of fruit, measured in utils per unit.
* $p_A,p_O>0$: constant fruit prices, measured in USD per unit.
* $I>0$: the available budget, measured in USD.

We assume that enough fruit is available for any purchase we can afford. We maximize total utility subject to the budget and nonnegativity constraints:

> __Fruit-allocation model:__
>
> Choose the fruit quantities to solve:
> $$
> \begin{aligned}
> \underset{x_A,x_O}{\text{maximize}}\quad & U(x_A,x_O)=u_Ax_A+u_Ox_O,\\
> \text{subject to}\quad & p_Ax_A+p_Ox_O\leq I,\\
> & x_A,x_O\geq 0.
> \end{aligned}
> $$
> The objective measures total utility in utils; the constraint limits expenditure in USD.

Let's store prices in `prices`, the budget in `budget`, and the three utility cases in `cases`. In each price or utility vector, the first entry refers to apples and the second to oranges:


In [2]:
# Set prices, budget, and utility cases -
# Every fruit vector is ordered as [apples, oranges].
prices = [2.0, 4.0] # dollars per unit: apples, oranges
budget = 100.0 # available budget, in dollars
cases = [
    (name = "A", utilities = [0.55, 0.45]),
    (name = "B", utilities = [0.15, 0.55]),
    (name = "C", utilities = [2.0, 4.0]),
]; # utility coefficients, in utility units per unit of fruit


### Why does utility per dollar determine the allocation?

Both fruits provide positive utility, so buying more fruit with unspent money would increase utility. An optimal allocation therefore uses the full budget. Solving the budget equation for the orange quantity gives:

$$
x_O=\frac{I-p_Ax_A}{p_O},\qquad 0\leq x_A\leq\frac{I}{p_A}.
$$

Substituting into the utility function gives utility along the budget boundary:

$$
\begin{aligned}
U\!\left(x_A,\frac{I-p_Ax_A}{p_O}\right)
&=u_Ax_A+\frac{u_O}{p_O}(I-p_Ax_A)\\
&=\frac{u_O}{p_O}I+p_A\left(\frac{u_A}{p_A}-\frac{u_O}{p_O}\right)x_A.
\end{aligned}
$$

The ratio $u_i/p_i$ measures utility per dollar spent on fruit $i\in\{A,O\}$. The first term is constant, so the coefficient of $x_A$ determines which allocation maximizes utility:

* If $u_A/p_A>u_O/p_O$, utility increases with $x_A$, so we buy only apples: $(x_A,x_O)=(I/p_A,0)$.
* If $u_A/p_A<u_O/p_O$, utility decreases with $x_A$, so we buy only oranges: $(x_A,x_O)=(0,I/p_O)$.
* If the ratios are equal, utility is constant along the budget boundary. Both corners and every full-budget mixture are optimal.

We use these relationships to calculate the predictions. Change `prices`, `budget`, or a case's `utilities` and rerun the parameter and prediction cells to update the table:


In [3]:
# Predict allocations from utility per dollar -
prediction_table = let
    table = DataFrame(); # one prediction for each utility case

    for case in cases
        ratios = case.utilities ./ prices; # utility gained per dollar
        # Choose the fruit with more utility per dollar; ties allow any full-budget mix.
        allocation = if ratios[1] > ratios[2]
            "$(budget / prices[1]) apples, 0 oranges"
        elseif ratios[2] > ratios[1]
            "0 apples, $(budget / prices[2]) oranges"
        else
            "Any mix spending $(budget) dollars"
        end

        # Store this case's ratios, allocation, and predicted maximum utility.
        push!(table, (
            case = case.name,
            apples = ratios[1],
            oranges = ratios[2],
            allocation = allocation,
            utility = budget * maximum(ratios), # USD × best utility per USD [utils]
        ));
    end

    table; # return the table from this local calculation block
end;


Let's display the predictions using [the pretty_table(...) function](https://ronisbr.github.io/PrettyTables.jl/stable/lib/library/#PrettyTables.pretty_table). The two ratio columns report utility per dollar, and the final column gives the predicted total utility:


In [4]:
# Display analytical predictions before comparing them with solver results -
pretty_table(HTML, prediction_table;
    column_labels = [
        "Case", "Apples: utility/dollar", "Oranges: utility/dollar",
        "Predicted allocation", "Total utility",
    ],
    alignment = [:l, :r, :r, :l, :r],
)


Case,Apples: utility/dollar,Oranges: utility/dollar,Predicted allocation,Total utility
A,0.275,0.1125,"50.0 apples, 0 oranges",27.5
B,0.075,0.1375,"0 apples, 25.0 oranges",13.75
C,1.0,1.0,Any mix spending 100.0 dollars,100.0


Compare the two ratios in each row. The larger ratio identifies which fruit receives the budget. Equal ratios allow either corner allocation or any mixture that spends the full budget, all with the same total utility. We will now compare these predictions with the solver results.


___

## Task 2: Solve the model and interpret the allocations

In this task, we solve each utility case and compare the results with our predictions. We then use the geometry of the budget and utility lines to explain the allocations.

[The solve_fruit_problem(...) function](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.solve_fruit_problem-Tuple%7BAbstractVector%7B%3C%3AReal%7D%2C%20AbstractVector%7B%3C%3AReal%7D%2C%20Real%7D) takes a utility vector, the price vector, and the budget. It builds and solves the linear program, checks that the solver reports an optimal solution, and returns the quantities, total utility, expenditure, and solver status. We collect these results in `comparison` alongside the predicted utility from Task 1:


In [5]:
# Solve each case with the same prices and budget -
# Each comprehension collects one result per case, preserving case order.
# Julia syntax: https://docs.julialang.org/en/v1/manual/arrays/#Comprehensions
solutions = [solve_fruit_problem(case.utilities, prices, budget) for case in cases];

# Assemble solver results alongside the predictions -
# Quantity index 1 denotes apples; index 2 denotes oranges.
comparison = DataFrame(
    case = [case.name for case in cases],
    apples = [solution.quantities[1] for solution in solutions],
    oranges = [solution.quantities[2] for solution in solutions],
    expenditure = [solution.expenditure for solution in solutions],
    utility = [solution.utility for solution in solutions],
    predicted_utility = prediction_table.utility,
    status = [string(solution.status) for solution in solutions],
);


Let's display the quantities and check whether the solver's utility agrees with our prediction. Expenditure should equal the budget because all utility coefficients are positive:


In [6]:
# Display purchases, expenditure, and achieved versus predicted utility -
pretty_table(HTML, comparison;
    column_labels = [
        "Case", "Apples", "Oranges", "Spent (USD)",
        "Utility", "Predicted utility", "Status",
    ],
    alignment = [:l, :r, :r, :r, :r, :r, :l],
)


Case,Apples,Oranges,Spent (USD),Utility,Predicted utility,Status
A,50.0,0.0,100.0,27.5,27.5,OPTIMAL
B,0.0,25.0,100.0,13.75,13.75,OPTIMAL
C,0.0,25.0,100.0,100.0,100.0,OPTIMAL


For a case with unequal utility-per-dollar ratios, the solver should allocate the full budget to the preferred fruit. When the ratios are equal, the solver returns one of many optimal allocations; either corner or a mixture can achieve the predicted utility. Compare the utility values and expenditure before interpreting a particular quantity vector.

### How do the slopes explain the solution?

The following figure shows apples on the horizontal axis and oranges on the vertical axis. Let $m_I$ denote the budget-line slope and $m_O$ the slope of a line with fixed total utility $\bar U$; the subscript $O$ in $m_O$ refers to the objective. The two lines and their slopes are given by:

$$
\begin{aligned}
\text{Budget:}\quad x_O &= \frac{I}{p_O}-\frac{p_A}{p_O}x_A,
& m_I &= -\frac{p_A}{p_O},\\
\text{Fixed utility:}\quad x_O &= \frac{\bar U}{u_O}-\frac{u_A}{u_O}x_A,
& m_O &= -\frac{u_A}{u_O}.
\end{aligned}
$$

Increasing utility shifts the fixed-utility line outward. The optimal line is the last one that touches the feasible region. The figure illustrates how its contact with the budget boundary changes with the relative slopes:


![Three slope cases: an apple-only optimum, an orange-only optimum, and an optimal budget edge.](figs/Fig-ThreeCases-LP-Schematic.svg)


In **panel A**, $|m_O|>|m_I|$, so the optimal line touches the apple corner. In **panel B**, $|m_O|<|m_I|$, so it touches the orange corner. In **panel C**, the slopes are equal and the optimal line coincides with the budget edge, giving multiple optimal allocations.

The panels show geometric possibilities rather than plots recalculated from our inputs. After changing the parameters, a case may correspond to a different panel. The dashed blue stock limits lie beyond the budget intercepts; our model assumes sufficient stock and includes no inventory constraints.

The table and geometry explain the solver's choice. Next, we will check the returned quantities and recompute expenditure and utility independently.


___

## Task 3: Verify feasibility and optimality

In this task, we check the returned quantities against the model constraints and verify that they achieve the largest possible utility. These checks use the current prices, budget, and utility coefficients, so they remain applicable when we change the inputs.

First, the quantities must be nonnegative and their expenditure must respect the budget. To check optimality, let $r_{\max}=\max\{u_A/p_A,u_O/p_O\}$ be the largest utility-per-dollar ratio. For any feasible allocation, total utility satisfies:

$$
\begin{aligned}
U(x_A,x_O)
&=\frac{u_A}{p_A}p_Ax_A+\frac{u_O}{p_O}p_Ox_O\\
&\leq r_{\max}(p_Ax_A+p_Ox_O)\\
&\leq r_{\max}I.
\end{aligned}
$$

Spending the full budget on a fruit with ratio $r_{\max}$ attains this bound. Thus, a feasible solution with utility $r_{\max}I$ is optimal, whether or not its quantities are unique.

We recompute expenditure and utility from the returned quantities using [the dot(...) function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.dot). The checks compare these calculations with the reported values and the analytical bound. Because the utility coefficients are positive, we also check that the full budget was spent. Approximate comparisons allow for floating-point roundoff:


In [7]:
# Verify feasibility, reported values, and the analytical utility bound -
@testset "Fruit-allocation checks" begin
    quantity_tolerance = 1e-8; # allowed numerical roundoff, in fruit units

    # Pair each result with the utility coefficients used to solve it.
    for (case, solution) in zip(cases, solutions)
        quantities = solution.quantities;
        expenditure = dot(prices, quantities); # recomputed expenditure, in USD
        utility = dot(case.utilities, quantities); # recomputed utility, in utils
        utility_bound = budget * maximum(case.utilities ./ prices); # utils

        # Require finite, nonnegative purchases and full use of the budget.
        @test all(isfinite, quantities)
        @test all(quantities .>= -quantity_tolerance)
        @test expenditure ≈ budget

        # Compare independent calculations with the reported values.
        @test expenditure ≈ solution.expenditure
        @test utility ≈ solution.utility

        # Attaining the bound establishes optimality for this one-budget model.
        @test utility ≈ utility_bound
        @test solution.status == MathOptInterface.OPTIMAL
    end
end;


Test Summary:           | Pass  Total  Time
Fruit-allocation checks |   21     21  0.4s


A passing test summary confirms that the returned quantities are feasible and attain the analytical optimum within numerical tolerance. We check objective values rather than require one particular quantity vector, which matters when several allocations are optimal.


### Why can different allocations be equally good?

Suppose the two utility-per-dollar ratios are equal to a common value $r>0$. Let $\lambda\in[0,1]$ be the fraction of the budget spent on apples. The remaining fraction is spent on oranges, giving the allocation:

$$
\bigl(x_A(\lambda),x_O(\lambda)\bigr)
=\left(\frac{\lambda I}{p_A},\frac{(1-\lambda)I}{p_O}\right).
$$

Every value of $\lambda$ uses the full budget. Because both fruits provide the same utility per dollar, the resulting utility is given by:

$$
U(\lambda)=r\lambda I+r(1-\lambda)I=rI.
$$

The endpoints $\lambda=0$ and $\lambda=1$ purchase only oranges or only apples; intermediate values purchase both. All these allocations have the same optimal utility. A solver can return any one of them, so different quantity vectors can both be correct. If we change the coefficients so the ratios differ, this freedom disappears and the preferred corner becomes the unique optimum.


___

## Summary

We connected a resource-allocation model with analytical predictions, solver results, and independent checks.

> __Key Takeaways:__
>
> * **Formulating the allocation problem:** We chose continuous fruit quantities as decision variables and maximized linear utility subject to a budget and nonnegativity constraints. Stating the units and assumptions connected the purchasing question to its mathematical model.
> * **Predicting and explaining the optimum:** We compared utility per dollar to predict the preferred fruit and used the budget and utility slopes to explain the geometry. Equal ratios produced an entire edge of optimal allocations with the same utility.
> * **Verifying the result:** We recomputed expenditure and utility from the returned quantities and checked an analytical upper bound. This established feasibility and optimality without requiring a particular allocation when multiple optima exist.

For this model, we spend the budget on the fruit that provides more utility per dollar. If both ratios are equal, any combination of apples and oranges that spends the full budget gives the same maximum utility.
